# Análise Comparativa de Chat Live
Este notebook gera visualizações para todos os vídeos configurados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

# 1. Carregar Configs
with open('config/settings.json', 'r') as f:
    videos_config = json.load(f)
with open('config/video_info.json', 'r') as f:
    videos_info = {v['video_id']: v['title'] for v in json.load(f)}

def t2s(t): 
    parts = list(map(int, t.split(':')))
    return parts[0]*3600 + parts[1]*60 + parts[2] if len(parts)==3 else parts[0]*60 + parts[1]

def plot_video_analysis(v_config):
    v_id = v_config['video_id']
    v_title = videos_info.get(v_id, v_id)
    csv_path = f'data/processed/chat_{v_id}_processed.csv'
    
    if not os.path.exists(csv_path): return
    
    df = pd.read_csv(csv_path)
    v_start = t2s(v_config['game_start_time'])
    fh_end = (t2s(v_config['first_half_end']) - v_start) / 60
    sh_start = (t2s(v_config['second_half_start']) - v_start) / 60
    sh_end = (t2s(v_config['second_half_end']) - v_start) / 60

    df['min_lin'] = (df['timestamp_jogo_segundos'] // 60).astype(int)
    vol = df.groupby('min_lin').size().reindex(range(int(df['min_lin'].min()), int(df['min_lin'].max()) + 1), fill_value=0)
    
    # Suavizado
    vol_s = vol.rolling(window=5, center=True).mean()
    diff_s = vol_s.diff()
    spikes = vol_s[diff_s >= 5]

    plt.figure(figsize=(15, 6))
    plt.plot(vol_s.index, vol_s.values, color='darkblue', linewidth=2, label='Tendência (Média Móvel)')
    plt.plot(vol.index, vol.values, color='gray', alpha=0.15, label='Bruto')
    
    plt.axvspan(0, fh_end, color='lightgreen', alpha=0.15, label='1º Tempo')
    plt.axvspan(sh_start, sh_end, color='lightblue', alpha=0.15, label='2º Tempo')
    
    plt.scatter(spikes.index, spikes.values, color='magenta', s=40, zorder=5)
    for m, v in spikes.items():
        label = df[df['min_lin'] == m]['label_partida'].iloc[0] if m in df['min_lin'].values else f"{int(m)}'"
        plt.annotate(label, (m, v), textcoords="offset points", xytext=(0,10), ha='center', color='magenta', fontsize=9, fontweight='bold')

    plt.title(f'{v_title}\n({v_id})', fontsize=12)
    plt.xlabel('Minutos Lineares')
    plt.ylabel('Mensagens/min')
    plt.legend(loc='upper left', fontsize='small')
    plt.grid(True, alpha=0.1)
    plt.show()

for config in videos_config:
    plot_video_analysis(config)